# Exploración de datos de Instacart con Python y Pandas

**Actividad:** GA1-220501094-01-AA1-EV04 · Fundamentos en Pandas  
**Aprendiz:** Carlos Eduardo Herrera Sibaja  
**Programa:** Análisis y Desarrollo de Software  
**Fecha:** 28/10/2026

## Propósito y comprensión del problema

Instacart registra pedidos, clientes, productos y la clasificación de su catálogo. El equipo de análisis necesita reconocer la estructura y calidad básica de esos datos y practicar consultas con Pandas antes de plantear análisis más profundos. En este notebook se cargan los cinco archivos CSV, se inspeccionan sus dimensiones y tipos, se consultan subconjuntos mediante selección y filtros, y se calculan indicadores descriptivos.

Explorar antes de analizar permite comprobar que los archivos se leyeron correctamente, conocer el significado y tipo de las columnas, detectar datos ausentes y escoger operaciones adecuadas. Así se reducen errores de interpretación; por ejemplo, una fila de `order_products` representa un artículo dentro de un pedido, no un pedido ni un cliente.

### Archivos y relaciones conceptuales

| Archivo | Información | Campos de relación principales |
|---|---|---|
| `instacart_orders.csv` | Pedidos, usuario, número de compra, día, hora e intervalo desde el pedido anterior. | `order_id` identifica un pedido; `user_id` identifica un usuario. |
| `products.csv` | Catálogo de productos y su clasificación. | `product_id` identifica el artículo; `aisle_id` y `department_id` enlazan con las tablas de clasificación. |
| `aisles.csv` | Nombres de los pasillos. | `aisle_id`. |
| `departments.csv` | Nombres de los departamentos. | `department_id`. |
| `order_products.csv` | Artículos que aparecen en cada pedido, posición en el carrito y recompra. | `order_id` enlaza con pedidos; `product_id` con productos. |

Las claves compartidas permiten relacionar conceptualmente las tablas. Este ejercicio se concentra en la exploración básica y no necesita unirlas para responder los filtros propuestos.

### Preguntas de análisis

1. ¿A qué hora se registran más pedidos y cuál es el promedio de hora del día?
2. ¿Cuántos artículos aparecen como recompra (`reordered = 1`) y qué proporción representan de las filas de artículos?
3. ¿Cuántos pedidos tienen número de orden superior a 10 y cuántos de ellos se realizaron a las 10:00?


### Fuentes de trabajo

- [Materiales de ciencia de datos (Notion)](https://educated-show-144.notion.site/Materiales-de-ciencia-de-datos-3fe4671e02a1807aa6cae213e826c33b): apartados sobre Pandas, DataFrames, lectura de CSV, indexación lógica y conteos/sumas.
- [Carpeta de datasets de Instacart (Google Drive)](https://drive.google.com/drive/folders/19vYGYdE0hI1DzvpBcCZskzJ0u2TZjwas): archivos CSV usados en la actividad.



## Momento 2 · Importación y exploración inicial

Los CSV usan punto y coma como separador. Los archivos deben permanecer junto a este notebook o ajustarse las rutas de `read_csv`. `head()` y `tail()` muestran cinco filas de cada tabla; las dimensiones informan filas y columnas.

In [1]:
from pathlib import Path
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
candidatos = [NOTEBOOK_DIR / "data", NOTEBOOK_DIR.parent / "data"]
DATA_DIR = next((ruta for ruta in candidatos if ruta.is_dir()), NOTEBOOK_DIR)
orders = pd.read_csv(DATA_DIR / "instacart_orders.csv", sep=";")
products = pd.read_csv(DATA_DIR / "products.csv", sep=";")
aisles = pd.read_csv(DATA_DIR / "aisles.csv", sep=";")
departments = pd.read_csv(DATA_DIR / "departments.csv", sep=";")
order_products = pd.read_csv(DATA_DIR / "order_products.csv", sep=";")

dataframes = {
    "orders": orders,
    "products": products,
    "aisles": aisles,
    "departments": departments,
    "order_products": order_products,
}
print("Archivos cargados:")
for name, df in dataframes.items():
    print(f"{name}: {df.shape[0]:,} filas × {df.shape[1]} columnas")

Archivos cargados:
orders: 478,967 filas × 6 columnas
products: 49,694 filas × 4 columnas
aisles: 134 filas × 2 columnas
departments: 21 filas × 2 columnas
order_products: 4,545,007 filas × 4 columnas


In [2]:
for name, df in dataframes.items():
    print(f"\n{name} · primeros cinco registros")
    print(df.head().to_string(index=False))
    print(f"\n{name} · últimos cinco registros")
    print(df.tail().to_string(index=False))


orders · primeros cinco registros
 order_id  user_id  order_number  order_dow  order_hour_of_day  days_since_prior_order
  1515936   183418            11          6                 13                    30.0
  1690866   163593             5          5                 12                     9.0
  1454967    39980             4          5                 19                     2.0
  1768857    82516            56          0                 20                    10.0
  3007858   196724             2          4                 12                    17.0

orders · últimos cinco registros
 order_id  user_id  order_number  order_dow  order_hour_of_day  days_since_prior_order
  3210681     5617             5          1                 14                     7.0
  3270802   112087             2          3                 13                     6.0
   885349    82944            16          2                 11                     6.0
   216274     4391             3          3                  

### Respuestas de exploración

- `order_products` contiene más registros: 4.545.007 filas. Representa los artículos vinculados a los pedidos.
- `aisles` y `departments` contienen menos columnas: dos cada uno.
- `orders` describe eventos de compra y usuarios; `products` describe artículos del catálogo y sus categorías. Sus filas tienen unidades de análisis diferentes.
- Conocer las dimensiones ayuda a estimar el volumen, revisar que la lectura coincida con lo esperado y evitar confundir filas con personas o pedidos.

## Momento 3 · Estructura y calidad inicial

`columns` muestra los campos, `dtypes` su tipo inferido, `shape` las dimensiones e `info()` resume tipos y valores no nulos. `int64` representa enteros, `float64` números con decimales (y puede representar ausencias) y `object` suele representar texto o valores mixtos. Algunas versiones recientes de Pandas pueden mostrar columnas de texto como `str` o `string` en vez de `object`.


In [3]:
for name, df in dataframes.items():
    print(f"\n===== {name} =====")
    print("Columnas:", list(df.columns))
    print("Tipos:")
    print(df.dtypes.to_string())
    print("Resumen info():")
    df.info()


===== orders =====
Columnas: ['order_id', 'user_id', 'order_number', 'order_dow', 'order_hour_of_day', 'days_since_prior_order']
Tipos:
order_id                    int64
user_id                     int64
order_number                int64
order_dow                   int64
order_hour_of_day           int64
days_since_prior_order    float64
Resumen info():
<class 'pandas.DataFrame'>
RangeIndex: 478967 entries, 0 to 478966
Data columns (total 6 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   order_id                478967 non-null  int64  
 1   user_id                 478967 non-null  int64  
 2   order_number            478967 non-null  int64  
 3   order_dow               478967 non-null  int64  
 4   order_hour_of_day       478967 non-null  int64  
 5   days_since_prior_order  450148 non-null  float64
dtypes: float64(1), int64(5)
memory usage: 21.9 MB

===== products =====
Columnas: ['product_id', 'product_name'

In [4]:
diagnostico = pd.DataFrame([
    {
        "archivo": name,
        "filas": df.shape[0],
        "columnas": df.shape[1],
        "valores_ausentes": int(df.isna().sum().sum()),
        "observaciones": "; ".join(
            f"{col}: {int(n)} ausentes" for col, n in df.isna().sum().items() if n > 0
        ) or "Sin valores ausentes detectados",
    }
    for name, df in dataframes.items()
])
print(diagnostico.to_string(index=False))

       archivo   filas  columnas  valores_ausentes                          observaciones
        orders  478967         6             28819 days_since_prior_order: 28819 ausentes
      products   49694         4              1258            product_name: 1258 ausentes
        aisles     134         2                 0        Sin valores ausentes detectados
   departments      21         2                 0        Sin valores ausentes detectados
order_products 4545007         4               836        add_to_cart_order: 836 ausentes


### Lectura del diagnóstico

Se observan ausencias en `orders.days_since_prior_order`, `products.product_name` y `order_products.add_to_cart_order`. En pedidos, el intervalo puede no estar disponible para la primera compra del usuario; no se debe asumir que todo nulo es un error. Los nombres vacíos dificultan identificar o presentar algunos productos. Las posiciones vacías impiden estudiar el orden de incorporación al carrito en esas filas. Estas limitaciones deben considerarse antes de calcular indicadores sobre esas columnas. En esta fase no se eliminan ni corrigen registros.

`info()` aporta, además de las dimensiones, los tipos de datos y el conteo de valores no nulos por columna. Al comparar ese conteo con el total de filas se detectan columnas incompletas.


## Momento 4 · Indexación y filtrado lógico

Seleccionar columnas reduce el conjunto de campos; filtrar filas conserva las columnas pero limita los registros que cumplen una condición. `loc[]` consulta filas por condición o etiqueta y columnas por nombre, haciendo explícita la selección. Al combinar condiciones con `&` se retienen solo filas que cumplen ambas; cada condición debe ir entre paréntesis.


In [5]:
# 1. Seleccionar únicamente product_id y product_name.
productos_seleccionados = products.loc[:, ["product_id", "product_name"]]
print("Columnas:", productos_seleccionados.columns.tolist(), "| filas:", len(productos_seleccionados))
print(productos_seleccionados.head().to_string(index=False))

# 2. Productos cuyo department_id sea 1.
productos_depto_1 = products.loc[products["department_id"] == 1]
print("\nProductos del departamento 1:", len(productos_depto_1))
print(productos_depto_1.head().to_string(index=False))

Columnas: ['product_id', 'product_name'] | filas: 49694
 product_id                                                      product_name
          1                                        Chocolate Sandwich Cookies
          2                                                  All-Seasons Salt
          3                              Robust Golden Unsweetened Oolong Tea
          4 Smart Ones Classic Favorites Mini Rigatoni With Vodka Cream Sauce
          5                                         Green Chile Anytime Sauce

Productos del departamento 1: 4008
 product_id                                                      product_name  aisle_id  department_id
          4 Smart Ones Classic Favorites Mini Rigatoni With Vodka Cream Sauce        38              1
          8                                 Cut Russet Potatoes Steam N' Mash       116              1
         12                                        Chocolate Fudge Layer Cake       119              1
         18                 

In [6]:
# 3. Pedidos hechos a las 10 horas.
pedidos_10 = orders.loc[orders["order_hour_of_day"] == 10]
print("Pedidos a las 10:00:", len(pedidos_10))
print(pedidos_10.head().to_string(index=False))

# 4. Pedidos con order_dow igual a 0.
pedidos_dow_0 = orders.loc[orders["order_dow"] == 0]
print("\nPedidos con order_dow = 0:", len(pedidos_dow_0))
print(pedidos_dow_0.head().to_string(index=False))

Pedidos a las 10:00: 40578
 order_id  user_id  order_number  order_dow  order_hour_of_day  days_since_prior_order
   294269   107523            29          2                 10                     1.0
   133707   182261             1          3                 10                     NaN
  3378084   127629            21          1                 10                    20.0
  2161799    87345             7          0                 10                    24.0
  1739741   118405            48          2                 10                     3.0

Pedidos con order_dow = 0: 84090
 order_id  user_id  order_number  order_dow  order_hour_of_day  days_since_prior_order
  1768857    82516            56          0                 20                    10.0
  1843871    20084             5          0                 11                    15.0
  2521283    97711             2          0                 18                    22.0
  3406282    19356            21          0                  0       

In [7]:
# 5. Pedidos con número de orden superior a 10.
pedidos_mayor_10 = orders.loc[orders["order_number"] > 10]
print("Pedidos con order_number > 10:", len(pedidos_mayor_10))
print(pedidos_mayor_10.head().to_string(index=False))

# 6. Pedidos con más de 15 días desde el pedido anterior.
pedidos_intervalo_mayor_15 = orders.loc[orders["days_since_prior_order"] > 15]
print("\nPedidos con days_since_prior_order > 15:", len(pedidos_intervalo_mayor_15))
print(pedidos_intervalo_mayor_15.head().to_string(index=False))

Pedidos con order_number > 10: 244072
 order_id  user_id  order_number  order_dow  order_hour_of_day  days_since_prior_order
  1515936   183418            11          6                 13                    30.0
  1768857    82516            56          0                 20                    10.0
  2511640   148233            40          3                  8                     2.0
  2266494    34954            40          5                 15                     4.0
  1162566    43738            33          4                 11                     2.0

Pedidos con days_since_prior_order > 15: 110706
 order_id  user_id  order_number  order_dow  order_hour_of_day  days_since_prior_order
  1515936   183418            11          6                 13                    30.0
  3007858   196724             2          4                 12                    17.0
  2521283    97711             2          0                 18                    22.0
  2843219    74699             4          6

In [8]:
# 7. Filas de artículos que indican recompra.
articulos_recomprados = order_products.loc[order_products["reordered"] == 1]
print("Filas con reordered = 1:", len(articulos_recomprados))
print(articulos_recomprados.head().to_string(index=False))

# 8. Artículos incorporados en la primera posición del carrito.
articulos_primera_posicion = order_products.loc[order_products["add_to_cart_order"] == 1]
print("\nFilas con add_to_cart_order = 1:", len(articulos_primera_posicion))
print(articulos_primera_posicion.head().to_string(index=False))

Filas con reordered = 1: 2683838
 order_id  product_id  add_to_cart_order  reordered
   567889        1560                1.0          1
  2261212       26683                1.0          1
   491251        8670               35.0          1
  2571142        1940                5.0          1
  2456893       21616                4.0          1



Filas con add_to_cart_order = 1: 450046
 order_id  product_id  add_to_cart_order  reordered
   567889        1560                1.0          1
  2261212       26683                1.0          1
  1961225       37553                1.0          1
   639939       10017                1.0          1
   750040        8518                1.0          0


In [9]:
# Dos condiciones simultáneas en pedidos.
pedidos_10_y_mayor_10 = orders.loc[
    (orders["order_hour_of_day"] == 10) & (orders["order_number"] > 10)
]
print("Pedidos a las 10:00 y con order_number > 10:", len(pedidos_10_y_mayor_10))
print(pedidos_10_y_mayor_10.head().to_string(index=False))

# Dos condiciones simultáneas en productos.
productos_depto_1_pasillo_mayor_50 = products.loc[
    (products["department_id"] == 1) & (products["aisle_id"] > 50)
]
print("\nProductos del departamento 1 con aisle_id > 50:", len(productos_depto_1_pasillo_mayor_50))
print(productos_depto_1_pasillo_mayor_50.head().to_string(index=False))

Pedidos a las 10:00 y con order_number > 10: 21537
 order_id  user_id  order_number  order_dow  order_hour_of_day  days_since_prior_order
   294269   107523            29          2                 10                     1.0
  3378084   127629            21          1                 10                    20.0
  1739741   118405            48          2                 10                     3.0
    60724   110808            78          0                 10                     2.0
  3156095   113920            14          2                 10                     2.0

Productos del departamento 1 con aisle_id > 50: 1618
 product_id                        product_name  aisle_id  department_id
          8   Cut Russet Potatoes Steam N' Mash       116              1
         12          Chocolate Fudge Layer Cake       119              1
         18 Pizza for One Suprema  Frozen Pizza        79              1
         46      Raisin Cinnamon Bagels 5 count        58              1
        

### Interpretación de los filtros

Los filtros reducen la cantidad de filas, y las dos condiciones simultáneas son más restrictivas que cada condición por separado. Los resultados presentados muestran el conteo completo y una vista de cinco ejemplos para mantener el notebook legible. Los operadores lógicos deben usarse con paréntesis: `&` significa “y” (ambas condiciones) y `|` significa “o” (al menos una condición). Para comparar valores se usa `==`, mientras que `=` asigna un valor.


## Momento 5 · Indicadores básicos

Los conteos de pedidos y productos corresponden a filas de sus archivos. El archivo de artículos contiene varias filas por pedido. El promedio de `days_since_prior_order` ignora automáticamente los valores ausentes; por eso su cantidad válida es menor que la cantidad total de pedidos. La suma de `reordered` coincide con el conteo de filas con valor 1 porque la columna solo contiene 0 y 1.


In [10]:
metricas = pd.Series({
    "Registros de pedidos": orders.shape[0],
    "Registros del catálogo de productos": products.shape[0],
    "Promedio de order_number": orders["order_number"].mean(),
    "Promedio de days_since_prior_order (valores disponibles)": orders["days_since_prior_order"].mean(),
    "Artículos con reordered = 1": int((order_products["reordered"] == 1).sum()),
    "Suma de reordered": int(order_products["reordered"].sum()),
})
for pregunta, valor in metricas.items():
    print(f"{pregunta}: {valor:,.4f}" if isinstance(valor, float) else f"{pregunta}: {valor:,}")

print("\nEstadísticas de order_hour_of_day:")
print(orders["order_hour_of_day"].describe().to_string())
print("\nEstadísticas de add_to_cart_order:")
print(order_products["add_to_cart_order"].describe().to_string())

Registros de pedidos: 478,967.0000
Registros del catálogo de productos: 49,694.0000
Promedio de order_number: 17.1578
Promedio de days_since_prior_order (valores disponibles): 11.1018
Artículos con reordered = 1: 2,683,838.0000
Suma de reordered: 2,683,838.0000

Estadísticas de order_hour_of_day:
count    478967.000000
mean         13.446676
std           4.224986
min           0.000000
25%          10.000000
50%          13.000000
75%          16.000000
max          23.000000

Estadísticas de add_to_cart_order:


count    4.544171e+06
mean     8.351669e+00
std      7.080368e+00
min      1.000000e+00
25%      3.000000e+00
50%      6.000000e+00
75%      1.100000e+01
max      6.400000e+01


In [11]:
# Respuesta de las tres preguntas de análisis formuladas al inicio.
hora_mas_pedidos = int(orders["order_hour_of_day"].mode().iloc[0])
pedidos_hora_mas_frecuente = int((orders["order_hour_of_day"] == hora_mas_pedidos).sum())
recompras = int((order_products["reordered"] == 1).sum())
filas_articulos = len(order_products)
pedidos_numero_mayor_10 = int((orders["order_number"] > 10).sum())
pedidos_a_las_10_mayores_10 = len(pedidos_10_y_mayor_10)
print(f"1) Hora con más pedidos: {hora_mas_pedidos}:00 ({pedidos_hora_mas_frecuente:,} pedidos); promedio de hora: {orders['order_hour_of_day'].mean():.2f}.")
print(f"2) Recompras: {recompras:,} de {filas_articulos:,} filas ({recompras / filas_articulos:.1%}).")
print(f"3) Pedidos con order_number > 10: {pedidos_numero_mayor_10:,}; de estos, a las 10:00: {pedidos_a_las_10_mayores_10:,}.")

1) Hora con más pedidos: 10:00 (40,578 pedidos); promedio de hora: 13.45.
2) Recompras: 2,683,838 de 4,545,007 filas (59.1%).
3) Pedidos con order_number > 10: 244,072; de estos, a las 10:00: 21,537.


In [12]:
# Tabla de pregunta, valor obtenido e interpretación.
resumen = pd.DataFrame([
    ["¿Cuántos registros de pedidos contiene el archivo?", f"{len(orders):,}", "Son filas de pedidos registradas; no equivalen a usuarios únicos."],
    ["¿Cuántos registros de productos contiene el catálogo?", f"{len(products):,}", "Cada fila del catálogo corresponde a un producto."],
    ["¿Cuál es el promedio de order_number?", f"{orders['order_number'].mean():.2f}", "Promedio del número secuencial de pedido por usuario dentro de las filas disponibles."],
    ["¿Cuál es el promedio de días desde el pedido anterior?", f"{orders['days_since_prior_order'].mean():.2f}", f"Calculado sobre {orders['days_since_prior_order'].count():,} valores no ausentes."],
    ["¿Cuántas filas tienen reordered = 1?", f"{int((order_products['reordered'] == 1).sum()):,}", "Son artículos marcados como recompra, no pedidos ni clientes."],
    ["¿Cuál es la suma de reordered?", f"{int(order_products['reordered'].sum()):,}", "Al ser una variable 0/1, la suma cuenta las filas con valor 1."],
])
resumen.columns = ["Pregunta", "Valor obtenido", "Interpretación"]
print(resumen.to_string(index=False))

                                              Pregunta Valor obtenido                                                                        Interpretación
    ¿Cuántos registros de pedidos contiene el archivo?        478,967                     Son filas de pedidos registradas; no equivalen a usuarios únicos.
 ¿Cuántos registros de productos contiene el catálogo?         49,694                                     Cada fila del catálogo corresponde a un producto.
                 ¿Cuál es el promedio de order_number?          17.16 Promedio del número secuencial de pedido por usuario dentro de las filas disponibles.
¿Cuál es el promedio de días desde el pedido anterior?          11.10                                          Calculado sobre 450,148 valores no ausentes.
                  ¿Cuántas filas tienen reordered = 1?      2,683,838                         Son artículos marcados como recompra, no pedidos ni clientes.
                        ¿Cuál es la suma de reordered?      2,68

## Momento 6 · Resolución y conclusiones

### Síntesis

Python y Pandas permiten cargar estos CSV con `read_csv(sep=";")`, inspeccionar su estructura y consultar subconjuntos mediante selección de columnas, `loc[]` y máscaras booleanas. En esta ejecución, los cinco DataFrames tienen tamaños distintos porque describen entidades diferentes: pedidos, artículos, catálogo y categorías. Los filtros localizan ejemplos concretos y los métodos `mean()`, `sum()` y `describe()` resumen campos numéricos.

### Hallazgos principales

- Hay 478.967 filas de pedidos y 49.694 filas en el catálogo.
- `order_products` contiene 4.545.007 filas de artículos; 2.683.838 están marcadas como recompra (aprox. 59,1%).
- El `order_number` promedio es aproximadamente 17,16. El intervalo medio entre pedidos disponibles es aproximadamente 11,10 días.
- La hora promedio de los pedidos es aproximadamente 13,45; `describe()` permite ver su mediana, cuartiles, mínimo y máximo. `add_to_cart_order` tiene mediana 6 y máximo 64 entre sus valores disponibles.
- Hay valores ausentes en tres campos; en particular, el promedio del intervalo usa solo registros no nulos.

### Tres conclusiones

1. El conjunto `order_products` domina el volumen, por lo que conviene prestar atención a memoria y a la diferencia entre una fila de artículo y una fila de pedido.
2. La recompra es frecuente en los registros de artículos: alrededor del 59,1% tiene `reordered = 1`. El resultado describe líneas de artículos y no usuarios únicos.
3. Los filtros simultáneos permiten responder preguntas más específicas: por ejemplo, contar pedidos a las 10:00 que además tienen `order_number > 10`. La interpretación depende de entender las claves y el nivel de detalle de cada tabla.

### Limitaciones

Este análisis es exploratorio y descriptivo. No demuestra causas ni representa necesariamente todos los clientes o periodos de Instacart. El archivo de pedidos no proporciona aquí nombres de días; por ello `order_dow = 0` se reporta con su código, sin asignar un día de la semana. No se deduplicaron filas, no se imputaron valores ausentes y no se estimó el número de usuarios únicos. Para análisis posteriores se deben validar relaciones entre tablas y documentar cualquier tratamiento de datos ausentes.

### Cierre

Las dimensiones, los tipos, `info()`, los filtros lógicos y los estadísticos básicos permiten reconocer qué información contienen los datos y consultar patrones iniciales de pedidos y artículos. Esta exploración constituye una base para análisis posteriores con uniones entre tablas y preguntas de negocio más específicas.
